In [1]:
import os
import json
import joblib
import pandas as pd

In [2]:
path_models_old = "data\\model\\model_extra_feature2"

In [3]:
SAVE_DIR = "optuna_artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)

In [4]:
base_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",

    "learning_rate": 0.06,
    "n_estimators": 6000,

    "num_leaves": 95,
    "max_depth": 9,

    "min_child_samples": 180,
    "min_split_gain": 0.01,

    "subsample": 0.75,
    "subsample_freq": 1,
    "colsample_bytree": 0.35,

    "reg_alpha": 2.0,
    "reg_lambda": 12.0,

    "max_bin": 127,

    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}

In [5]:
path_train_target = "data\\train\\train_target.parquet"
train_t = pd.read_parquet(path_train_target)
target_cols = [col for col in train_t.columns if col != "customer_id"]

In [10]:
def feature_importance(model):
    summ_imp = model.feature_importances_.sum()

    if summ_imp == 0:
        feat_imp_df = pd.DataFrame({
            "feature": model.feature_name_,
            "importance": model.feature_importances_
        })
    else:
        feat_imp_df = pd.DataFrame({
            "feature": model.feature_name_,
            "importance": model.feature_importances_ / summ_imp
        })

    return feat_imp_df.sort_values("importance", ascending=False)


def get_feature_drop_from_old_model(target, path_models_old):
    model = joblib.load(os.path.join(path_models_old, f"{target}.pkl"))
    feature_imp_table = feature_importance(model)

    feature_drop = feature_imp_table.loc[
        feature_imp_table["importance"] == 0, "feature"
    ].tolist()

    return feature_drop



def build_base_meta(target, train_t, base_params, feature_drop):
    params = base_params.copy()

    y = train_t[target]

    meta = {
        "target": target,
        "status": "base",
        "source": "old_model_zero_importance + base_params",
        "best_auc": None,
        "best_iteration": None,
        "best_trial_number": None,
        "best_params": params,
        "feature_drop_count": len(feature_drop),
        "n_trials": 0,
    }
    return meta


def init_target_artifact(target, train_t, path_models_old, save_dir=SAVE_DIR):
    target_dir = os.path.join(save_dir, target)
    os.makedirs(target_dir, exist_ok=True)

    feature_drop = get_feature_drop_from_old_model(target, path_models_old)
    meta = build_base_meta(target, train_t, base_params, feature_drop)

    with open(os.path.join(target_dir, "feature_drop.json"), "w", encoding="utf-8") as f:
        json.dump(feature_drop, f, ensure_ascii=False, indent=2)

    with open(os.path.join(target_dir, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    return {
        "target": target,
        "target_dir": target_dir,
        "feature_drop_count": len(feature_drop),
    }


def init_all_targets_artifacts(target_cols, train_t, path_models_old, save_dir=SAVE_DIR):
    os.makedirs(save_dir, exist_ok=True)

    summary = {}

    for target in target_cols:
        info = init_target_artifact(
            target=target,
            train_t=train_t,
            path_models_old=path_models_old,
            save_dir=save_dir,
        )
        summary[target] = info
        print(
            f"{target}: "
            f"drop={info['feature_drop_count']}"

        )

    with open(os.path.join(save_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    return summary

In [11]:
summary = init_all_targets_artifacts(
    target_cols=target_cols,
    train_t=train_t,
    path_models_old=path_models_old,
    save_dir=SAVE_DIR,
)

target_1_1: drop=1594
target_1_2: drop=2262
target_1_3: drop=1569
target_1_4: drop=1522
target_1_5: drop=1975
target_2_1: drop=1739
target_2_2: drop=1421
target_2_3: drop=2398
target_2_4: drop=1797
target_2_5: drop=1946
target_2_6: drop=1976
target_2_7: drop=2283
target_2_8: drop=2636
target_3_1: drop=1403
target_3_2: drop=1470
target_3_3: drop=2500
target_3_4: drop=2008
target_3_5: drop=1863
target_4_1: drop=1866
target_5_1: drop=1877
target_5_2: drop=2482
target_6_1: drop=1788
target_6_2: drop=1687
target_6_3: drop=1639
target_6_4: drop=1752
target_6_5: drop=2161
target_7_1: drop=1361
target_7_2: drop=1373
target_7_3: drop=1925
target_8_1: drop=1259
target_8_2: drop=1446
target_8_3: drop=1599
target_9_1: drop=1946
target_9_2: drop=1526
target_9_3: drop=1746
target_9_4: drop=2136
target_9_5: drop=1810
target_9_6: drop=1309
target_9_7: drop=1381
target_9_8: drop=1815
target_10_1: drop=1129
